In [1]:
import pandas as pd
# ---- run me first in every notebook ----
import sys, os
from pathlib import Path

# Ensure we are at the project root (the folder that contains `src/`)
# If your notebook sits in the root, this is already correct.
root = Path.cwd()

print("Project root:", root)

# Get the directory of the current script
current_dir = os.getcwd()
# Get the parent directory
parent_dir = os.path.dirname(current_dir)
grandparent_dir = os.path.dirname(parent_dir)

# Add the parent directory to the system path
sys.path.append(parent_dir)
sys.path.append(grandparent_dir)
print(grandparent_dir)

Project root: d:\Bioinformatics\Rosaloid\data\round1
d:\Bioinformatics\Rosaloid


In [2]:
# round1_train_gp.py — train GP on NPZ train matrices
import numpy as np, joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C

METHODS = ["additive","plldelta","mutant_ctx"]
Path("cache/models").mkdir(parents=True, exist_ok=True)

for tag in METHODS:
    p = Path(Path.cwd().parent / "round0" / f"round0_{tag}_train_matrix.npz")
    if not p.exists():
        print(f"[{tag}] missing {p.name} — skip"); continue

    dat = np.load(p, allow_pickle=True)
    X0  = dat["X"].astype(np.float32)
    y   = dat["y_norm"].astype(np.float32)

    sc1 = StandardScaler().fit(X0);         X1 = sc1.transform(X0)
    pca = PCA(n_components=64, random_state=7).fit(X1)
    X2  = pca.transform(X1)
    sc2 = StandardScaler().fit(X2);         X  = sc2.transform(X2)

    d = X.shape[1]
    kernel = C(1.0, (1e-2, 1e2)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2,1e2), nu=2.5) \
             + WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-6, 1e-1))

    gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=False,
                                  n_restarts_optimizer=3, random_state=7)
    gp.fit(X, y)

    joblib.dump({"sc1":sc1,"pca":pca,"sc2":sc2,"gp":gp}, f"cache/models/round0_{tag}_gp.pkl")
    print(f"[{tag}] GP kernel -> {gp.kernel_}")


C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 4 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 6 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceW

[additive] GP kernel -> 7.55**2 * Matern(length_scale=[13, 4.23, 14.5, 4.73, 100, 13.4, 100, 19.7, 25.1, 100, 23.9, 12, 100, 12.1, 100, 9.26, 28, 100, 28.9, 100, 20.9, 14.5, 19.9, 13.4, 100, 100, 100, 100, 8.08, 100, 100, 73.2, 55.5, 100, 100, 100, 5.64, 49.4, 100, 100, 100, 100, 64.6, 100, 18.9, 100, 40.4, 100, 27.7, 30.4, 100, 100, 72.1, 100, 17.4, 100, 100, 61.4, 100, 100, 33.7, 100, 100, 13.5], nu=2.5) + WhiteKernel(noise_level=1e-06)
[plldelta] GP kernel -> 4.35**2 * Matern(length_scale=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], nu=2.5) + WhiteKernel(noise_level=1e-06)
[mutant_ctx] GP kernel -> 5.9**2 * Matern(length_scale=[1, 1, 1, 1.02, 1, 1, 1.02, 1.03, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1.01, 1, 1, 1.01, 1, 1, 1, 1, 1, 1, 1, 1.01, 1, 1.01, 1.01, 1, 1, 1, 1.01, 1, 1.01, 1, 1, 1, 1.01, 1, 1.01, 1.01, 1, 1, 1, 1.01, 1.02, 1, 1, 1,

C:\Users\thana\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-06. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
